In [ ]:
# ============================================================
# CELL 1 — Install packages
# This cell installs all the Python libraries we need:
#   - The project repo (semantic-correspondence)
#   - OpenCV for image processing
#   - tqdm for progress bars
#   - SAM (Segment Anything Model) from Meta
# You only need to run this once per Colab session.
# ============================================================

!test -d /content/semantic-correspondence || git clone -q https://github.com/aexomir/semantic-correspondence.git /content/semantic-correspondence
!pip -q install -U pip
!pip -q install -r /content/semantic-correspondence/requirements.txt
!pip -q install opencv-python tqdm
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# CELL 2 — Download DINO model code
# DINOv2 and DINOv3 are not on PyPI, so we clone their GitHub
# repos directly into this Colab environment.
# We also install the extra packages these repos require.
# ============================================================

!rm -rf dinov2 dinov3
!git clone -q https://github.com/facebookresearch/dinov2.git
!git clone -q https://github.com/facebookresearch/dinov3.git

!pip -q install omegaconf torchmetrics fvcore iopath submitit ftfy regex scikit-learn termcolor

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive and set file paths
# We store model weights and the dataset on Google Drive so
# they survive between sessions. This cell:
#   1. Imports all libraries used throughout the notebook
#   2. Connects to Google Drive
#   3. Defines paths to all model weight files (.pth)
#   4. Defines paths to the dataset files
#   5. Creates a local /content/data folder for fast disk access
# ============================================================

import os
import sys
import shutil
import types
import math
import json
import cv2
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from contextlib import nullcontext
import pandas as pd
from google.colab import drive

# Connect to Google Drive (will ask for permission the first time)
drive.mount('/content/drive', force_remount=True)

MYDRIVE = '/content/drive/MyDrive'

# Paths to saved model weights on Drive
DINOV2_WEIGHTS = os.path.join(MYDRIVE, 'dinov2_vitl14_pretrain_big.pth')
DINOV3_WEIGHTS = os.path.join(MYDRIVE, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')
SAM_WEIGHTS_B  = os.path.join(MYDRIVE, 'sam_vit_b_01ec64_small.pth')

# Paths to dataset files on Drive
SPAIR_DIR = os.path.join(MYDRIVE, 'SPair-71k')
SPAIR_TAR = os.path.join(MYDRIVE, 'SPair-71k.tar.gz')
PFW_ZIP   = os.path.join(MYDRIVE, 'pf-willow.zip')

# Local fast disk folder (on Colab's machine, not Drive)
LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

if not os.path.isdir(MYDRIVE):
    raise RuntimeError('Drive not mounted. Complete the auth prompt and re-run this cell.')

print('OK:', MYDRIVE)


Mounted at /content/drive
OK: /content/drive/MyDrive


In [ ]:
# ============================================================
# CELL 4 — Copy the SPair-71k dataset to local disk
# Extract the tar.gz file directly into the local data directory,
# or copy the directory if the tar file is missing.
# ============================================================
import shutil

local_spair = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')

if not os.path.exists(local_spair):
    if os.path.exists(SPAIR_TAR):
        print(f"Extracting {SPAIR_TAR} to {LOCAL_DATA_DIR}...")
        !tar -xzf {SPAIR_TAR} -C {LOCAL_DATA_DIR}
        print("Extraction complete!")
    elif os.path.exists(SPAIR_DIR):
        print(f"tar file not found. Copying directory {SPAIR_DIR} to {local_spair}...")
        shutil.copytree(SPAIR_DIR, local_spair)
        print("Copy complete!")
    else:
        print("Error: Neither tar file nor directory found in Drive.")
else:
    print(f"Dataset already exists at {local_spair}")

Extracting /content/drive/MyDrive/SPair-71k.tar.gz to /content/data...
Extraction complete!


In [ ]:
# ============================================================
# CELL 5 — Load the three pretrained models
# We load and compare three foundation models:
#   - DINOv2 ViT-L/14: self-supervised ViT from Meta (large)
#   - DINOv3 ViT-B/16: updated version of DINOv2
#   - SAM ViT-B: Segment Anything Model encoder from Meta (base)
# For each model we build the architecture, load saved weights
# from Google Drive, then move to GPU and set to eval mode.
# ============================================================

from segment_anything import SamPredictor, sam_model_registry

# Load SAM ViT-B
sam = sam_model_registry['vit_b'](checkpoint=SAM_WEIGHTS_B)
sam_predictor = SamPredictor(sam)

# Load DINOv2 ViT-L/14 (big weight file uses ViT-L architecture)
dinov2 = torch.hub.load('dinov2', 'dinov2_vitl14', source='local', weights=DINOV2_WEIGHTS)

# Load DINOv3 ViT-B/16 (small file — big file uses incompatible architecture)
dinov3 = torch.hub.load('dinov3', 'dinov3_vitb16', source='local', weights=DINOV3_WEIGHTS)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam.to(device).eval()
dinov2.to(device).eval()
dinov3.to(device).eval()

print('Ready. Device:', device)

# ── GPU sanity checks ────────────────────────────────────────────────────
print('DINOv2 device:', next(dinov2.parameters()).device)
print('DINOv3 device:', next(dinov3.parameters()).device)
print('SAM    device:', next(sam.parameters()).device)
assert str(next(dinov2.parameters()).device).startswith('cuda'), \
    'WARNING: DINOv2 is NOT on GPU! Check runtime type in Runtime > Change runtime type.'


/content/dinov2/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/content/dinov2/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/content/dinov2/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "file:///content/drive/MyDrive/dinov2_vitl14_pretrain_big.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain_big.pth


100%|██████████| 1.13G/1.13G [00:26<00:00, 46.3MB/s]


Downloading: "file:///content/drive/MyDrive/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth


100%|██████████| 327M/327M [00:08<00:00, 42.0MB/s]


Ready. Device: cuda
DINOv2 device: cuda:0
DINOv3 device: cuda:0
SAM    device: cuda:0


In [ ]:
# ============================================================
# CELL 6 — Feature extraction functions
# These three functions run an image through a model and
# return its internal feature map as a torch tensor.
# A 'feature map' is a grid of vectors — one vector per image
# patch — that describes what the model 'sees' in that region.
# ============================================================


def extract_dino_features(model, img_tensor, trainable=False):
    """
    Run an image through a DINO model and return its patch features.

    The model splits the image into a grid of small patches and produces
    one feature vector per patch. We reshape these vectors into a 2D
    spatial grid so they are easy to work with.

    Input:  img_tensor — a torch tensor of shape (1, C, H, W)
            trainable  — if True, keeps gradients (for finetuning);
                         if False, runs under no_grad and forces eval mode
    Output: torch tensor of shape (1, D, grid_H, grid_W)
              D = feature dimension, grid_H/W = number of patches per side
    """
    if not trainable:
        model.eval()

    ctx = torch.no_grad() if not trainable else nullcontext()
    with ctx:
        out = model.forward_features(img_tensor)
        # Different DINO versions store patch tokens under different key names
        if isinstance(out, dict):
            patch_tokens = out.get('x_norm_patchtokens', out.get('x_norm_patch_tokens'))
        else:
            patch_tokens = out

        if patch_tokens is None:
            keys = list(out.keys()) if isinstance(out, dict) else 'N/A'
            raise ValueError(f'Could not find patch tokens. Keys: {keys}')

        b, n, d = patch_tokens.shape
        grid = int(math.isqrt(n))  # num_patches must be a perfect square (e.g. 37*37=1369)
        if grid * grid != n:
            raise ValueError(f'Expected square number of patches, got N={n}')

        # Rearrange from (batch, num_patches, dim) -> (batch, dim, grid_H, grid_W)
        return patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)


def extract_dino_layers(model, img_tensor, layer_ids):
    """
    Run an image through DINO and return features from specific middle layers.

    Instead of just the final output, we can inspect features at any transformer
    block. This is useful when fine-tuning — you want to unfreeze only the last
    few layers and see which layers help the most.

    Input:  img_tensor — torch tensor (1, C, H, W)
            layer_ids  — list of layer indices to extract, e.g. [10, 11]
    Output: dict mapping each layer_id -> torch tensor (1, D, grid_H, grid_W)
    """
    model.eval()
    with torch.no_grad():
        # Get the output of every transformer block at once
        all_layers = model.get_intermediate_layers(img_tensor, n=len(model.blocks), norm=True)
        out = {}
        for lid in layer_ids:
            patch_tokens = all_layers[lid]
            b, n, d = patch_tokens.shape
            grid = int(math.isqrt(n))
            if grid * grid != n:
                raise ValueError(f'Layer {lid}: expected square patches, got N={n}')
            # Rearrange to (batch, dim, grid_H, grid_W)
            out[lid] = patch_tokens.permute(0, 2, 1).reshape(b, d, grid, grid)
        return out


def extract_sam_features(predictor, image_pil, res=1024):
    """Run an image through SAM's image encoder and return its feature map.

    To match SAM's pretrained setup (and avoid any positional-embedding
    interpolation), we run SAM at its native resolution: 1024×1024.

    Input:  image_pil — RGB PIL Image
            res       — must be 1024
    Output: torch tensor of shape (1, C, feat_H, feat_W)
    """
    if int(res) != 1024:
        raise ValueError(f"SAM without pos-embed resizing requires res=1024, got {res}")

    device = next(predictor.model.parameters()).device
    encoder = predictor.model.image_encoder

    # Resize image to 1024×1024
    image_resized = transforms.functional.resize(
        image_pil, (1024, 1024), interpolation=transforms.InterpolationMode.BILINEAR
    )

    # Convert to torch tensor (values in [0, 255]) and normalize using SAM's mean/std
    img_tensor = transforms.ToTensor()(image_resized).unsqueeze(0).to(device) * 255.0
    pixel_mean = torch.tensor([123.675, 116.28, 103.53], device=device).view(-1, 1, 1)
    pixel_std  = torch.tensor([58.395, 57.12, 57.375],  device=device).view(-1, 1, 1)
    img_tensor = (img_tensor - pixel_mean) / pixel_std

    with torch.no_grad():
        feats = encoder(img_tensor)

    return feats


In [ ]:
# ============================================================
# CELL 7 — Dataset paths, model registry, and feature cache
# This cell does three things:
#   1. Defines where all data and results live on disk
#   2. Lists all three models we want to test (MODEL_SPECS)
#   3. Defines helper functions to save/load feature files
#      as .pt (PyTorch) files so we never re-run models on
#      the same image twice
# ============================================================

# --- Dataset and output folder paths ---
SPAIR_ROOT = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')
JPEG_ROOT = os.path.join(SPAIR_ROOT, 'JPEGImages')              # folder with all the images
PAIRANN_TEST_ROOT = os.path.join(SPAIR_ROOT, 'PairAnnotation', 'test')  # folder with keypoint labels

# FIX: cache on local SSD (/content) instead of Google Drive.
# Drive I/O is 10–50× slower than local disk for thousands of small files.
# We sync to Drive once at the very end with sync_features_to_drive().
LOCAL_FEATURE_ROOT = '/content/features/spair71k'   # fast local SSD cache
DRIVE_FEATURE_ROOT = os.path.join(MYDRIVE, 'features', 'spair71k')  # persistent Drive copy
FEATURE_ROOT = LOCAL_FEATURE_ROOT   # all cache reads/writes go here
RESULTS_ROOT = os.path.join(MYDRIVE, 'results')                 # where we save evaluation results

# --- Model registry ---
# Each entry says: which model object to use, what image size it needs, and what type it is.
# 'dino' models use the torchvision preprocessing pipeline.
# 'sam' models use SAM's own preprocessing inside extract_sam_features.
MODEL_SPECS = {
    'dinov2_vitl14': {
        'model': dinov2,
        'img_size': (518, 518),   # 518 / 14 approx 37 patches per side
        'kind': 'dino',
    },
    'dinov3_vitb16': {
        'model': dinov3,
        'img_size': (512, 512),   # 512 / 16 = 32 patches per side
        'kind': 'dino',
    },
    'sam_vit_b_res1024': {
        'model': sam_predictor,
        'img_size': (1024, 1024),
        'kind': 'sam',
        'sam_res': 1024,
    },
}


def _ensure_parent_dir(path):
    """Create the folder that contains the given file path, if it does not exist yet."""
    os.makedirs(os.path.dirname(path), exist_ok=True)


def _rel_from_jpeg_root(image_path):
    """Return the path of an image relative to the dataset images folder (JPEG_ROOT)."""
    rel = os.path.relpath(image_path, JPEG_ROOT)
    if rel.startswith('..'):
        raise ValueError(f'Expected image under JPEG_ROOT={JPEG_ROOT}, got {image_path}')
    return rel


def feature_path(model_key, image_path, layer_id=None):
    """Build the file path where we store cached features for one image.

    The path mirrors the dataset folder structure so features are easy to find.
    Example: features/spair71k/dinov2_vitl14/dog/image001.pt
    """
    rel = _rel_from_jpeg_root(image_path)
    rel_no_ext = os.path.splitext(rel)[0]
    # If layer_id is provided (DINO intermediate layer), cache it separately
    layer_tag = '' if layer_id is None else f'_layer{int(layer_id)}'
    return os.path.join(FEATURE_ROOT, model_key, rel_no_ext + layer_tag + '.pt')


def save_feature_pt(path, feat_tensor):
    """Save a feature tensor to a .pt file. Creates the parent folder if needed."""
    _ensure_parent_dir(path)
    torch.save(feat_tensor, path)


def load_feature_pt(path):
    """Load and return the feature tensor stored in a .pt file."""
    return torch.load(path, map_location='cpu')


def dino_transform(img_size):
    """
    Build a torchvision preprocessing pipeline for DINO models.

    Three steps:
      1. Resize the image to img_size using bicubic interpolation
      2. Convert pixel values from [0, 255] integers to [0.0, 1.0] floats (ToTensor)
      3. Normalize with ImageNet mean and std (what DINO was trained with)

    Returns a torchvision Compose object — call it like a function on a PIL image.
    """
    return transforms.Compose(
        [
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


# Pre-build one transform pipeline per DINO model so we don't recreate it each time
_DINO_TRANSFORMS = {
    k: dino_transform(v['img_size'])
    for k, v in MODEL_SPECS.items()
    if v['kind'] == 'dino'
}


def compute_feature(model_key, image_path, layer_id=None):
    """
    Load one image, run it through the specified model, and return its feature map
    as a float16 torch tensor stored on CPU.

    For DINO models:
      - Apply the torchvision preprocessing pipeline (_DINO_TRANSFORMS)
      - Add a batch dimension and move to GPU
      - Run extract_dino_features -> get a torch tensor (1, D, H, W)

    For SAM:
      - Open image with PIL, convert to raw array for cv2
      - Run extract_sam_features -> get a torch tensor (1, C, H, W)

    Returns a float16 CPU tensor (D, H, W).
    Storing as float16 halves the disk space compared to float32.
    """
    spec = MODEL_SPECS[model_key]
    kind = spec['kind']

    img_pil = Image.open(image_path).convert('RGB')

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if kind == 'dino':
        t = _DINO_TRANSFORMS[model_key](img_pil).unsqueeze(0).to(device)  # (1, C, H, W)
        if layer_id is None:
            feat = extract_dino_features(spec['model'], t)                     # (1, D, H, W)
        else:
            feat = extract_dino_layers(spec['model'], t, [int(layer_id)])[int(layer_id)]  # (1, D, H, W)
    elif kind == 'sam':
        feat = extract_sam_features(spec['model'], img_pil, res=spec['sam_res'])  # (1, C, H, W)
    else:
        raise ValueError(f'Unknown kind: {kind}')

    # Remove batch dim, store as float16 on CPU to save memory
    return feat.squeeze(0).detach().half().cpu()


def load_or_compute_feature(model_key, image_path, layer_id=None):
    """
    Return the feature tensor for one image, using the disk cache when possible.

    - If a cached .pt file already exists on Drive -> load and return it immediately (fast)
    - If not -> run the model, save the result to Drive, then return it

    This means each image is processed by the model only once across all runs.
    """
    out_path = feature_path(model_key, image_path, layer_id=layer_id)
    if os.path.exists(out_path):
        return load_feature_pt(out_path)

    feat = compute_feature(model_key, image_path, layer_id=layer_id)
    save_feature_pt(out_path, feat)
    return feat


os.makedirs(FEATURE_ROOT, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)


def sync_features_to_drive():
    """
    Copy the local feature cache to Google Drive for persistence.
    Call this once after all features are computed / after fine-tuning.
    It only copies files that are missing on Drive, so it is safe to re-run.
    """
    print(f'Syncing {LOCAL_FEATURE_ROOT} -> {DRIVE_FEATURE_ROOT} ...')
    os.makedirs(DRIVE_FEATURE_ROOT, exist_ok=True)
    for root, dirs, files in os.walk(LOCAL_FEATURE_ROOT):
        rel = os.path.relpath(root, LOCAL_FEATURE_ROOT)
        dst_root = os.path.join(DRIVE_FEATURE_ROOT, rel)
        os.makedirs(dst_root, exist_ok=True)
        for fn in files:
            src_f = os.path.join(root, fn)
            dst_f = os.path.join(dst_root, fn)
            if not os.path.exists(dst_f):
                shutil.copy2(src_f, dst_f)
    print('Sync complete.')


def seed_local_cache_from_drive():
    """
    Copy any previously saved feature files from Drive to local SSD at startup.
    Run this once after mounting Drive so subsequent runs skip recomputation.
    """
    if not os.path.isdir(DRIVE_FEATURE_ROOT):
        print('No Drive cache found — starting fresh.')
        return
    print(f'Seeding local cache from {DRIVE_FEATURE_ROOT} ...')
    count = 0
    for root, dirs, files in os.walk(DRIVE_FEATURE_ROOT):
        rel = os.path.relpath(root, DRIVE_FEATURE_ROOT)
        dst_root = os.path.join(LOCAL_FEATURE_ROOT, rel)
        os.makedirs(dst_root, exist_ok=True)
        for fn in files:
            src_f = os.path.join(root, fn)
            dst_f = os.path.join(dst_root, fn)
            if not os.path.exists(dst_f):
                shutil.copy2(src_f, dst_f)
                count += 1
    print(f'Seeded {count} files into local cache.')


# Seed local SSD from Drive right away (fast copy, happens once per session)
seed_local_cache_from_drive()

print('Local cache root:', FEATURE_ROOT)
print('Drive  cache root:', DRIVE_FEATURE_ROOT)
print('Results root:', RESULTS_ROOT)
print('JPEG root:', JPEG_ROOT)
print('PairAnnotation test root:', PAIRANN_TEST_ROOT)


Seeding local cache from /content/drive/MyDrive/features/spair71k ...
Seeded 10002 files into local cache.
Local cache root: /content/features/spair71k
Drive  cache root: /content/drive/MyDrive/features/spair71k
Results root: /content/drive/MyDrive/results
JPEG root: /content/data/SPair-71k/JPEGImages
PairAnnotation test root: /content/data/SPair-71k/PairAnnotation/test


In [ ]:
# ============================================================
# CELL 8 — Pre-compute and cache features for all images
# Running a model on 1800 images every time we evaluate would
# be slow. Instead, we run each model once on all images and
# save the results as .pt (PyTorch) files on Google Drive.
# On future runs, we just load the saved files.
# ============================================================

# Only process files with these extensions
VALID_IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}


def iter_jpeg_images():
    """
    Walk through the dataset images folder and yield the full path
    of every valid image file (.jpg, .jpeg, .png).
    """
    for root, _, files in os.walk(JPEG_ROOT):
        for fn in files:
            ext = os.path.splitext(fn)[1].lower()
            if ext in VALID_IMAGE_EXTS:
                yield os.path.join(root, fn)


def precompute_features(model_keys=None, limit=None):
    """
    Run all images through each model and save their feature tensors to Drive.

    - model_keys: which models to run (default: all models in MODEL_SPECS)
    - limit: optionally process only the first N images (useful for quick tests)

    For each image, if the feature file already exists on Drive we skip it.
    This makes the function safe to re-run — it only processes new images.
    """
    if model_keys is None:
        model_keys = list(MODEL_SPECS.keys())

    images = list(iter_jpeg_images())
    if limit is not None:
        images = images[: int(limit)]

    print(f'Found {len(images)} images under {JPEG_ROOT}')
    print('Models:', model_keys)

    for model_key in model_keys:
        missing = 0
        for img_path in tqdm(images, desc=f'{model_key}: caching'):
            out_path = feature_path(model_key, img_path)
            if os.path.exists(out_path):
                continue  # already cached, skip

            feat = compute_feature(model_key, img_path)
            save_feature_pt(out_path, feat)
            missing += 1

        print(f'{model_key}: newly cached {missing} feature files')


# Example usage:
# precompute_features(['dinov2_vitb14'], limit=50)   # quick test on 50 images
# precompute_features(list(MODEL_SPECS.keys()))       # full run on all models


In [ ]:
# ============================================================
# CELL 9 — Evaluation functions (PCK metric)
# This cell defines the full evaluation pipeline.
# For each pair of images in the test set we:
#   1. Load the cached feature tensors for both images
#   2. For each labeled keypoint in the source image,
#      find the most similar patch in the target image
#   3. Check if the predicted location is close enough to
#      the ground-truth keypoint (measured by PCK)
# PCK = Percentage of Correct Keypoints
# A prediction counts as 'correct' if its distance from the
# ground truth is below a threshold x (object bounding box size)
# ============================================================

# We evaluate at four strictness levels: 5%, 10%, 15%, 20% of bbox size
PCK_THRESHOLDS = [0.05, 0.1, 0.15, 0.2]


def list_pair_files(split="test"):
    """
    Find all JSON annotation files for a given SPair-71k split.
    split: one of "test", "val", "trn"
    Each JSON file describes one image pair and contains the
    source/target keypoints and bounding box info.
    """
    split_dir = os.path.join(SPAIR_ROOT, 'PairAnnotation', split)
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f'Missing PairAnnotation {split} dir: {split_dir}')

    pair_files = []
    for root, _, files in os.walk(split_dir):
        for fn in files:
            if fn.endswith('.json'):
                pair_files.append(os.path.join(root, fn))

    pair_files.sort()
    return pair_files


# Keep the old name as a convenience alias so Stage 1 call sites still work
def list_test_pair_files():
    return list_pair_files(split="test")


def _parse_xy(p):
    """
    Safely read a keypoint coordinate from the annotation.
    Returns (x, y) as floats, or None if the point is missing,
    invalid, or has negative coordinates (means it was not labeled).
    """
    if p is None:
        return None
    if not isinstance(p, (list, tuple)):
        return None
    if len(p) < 2:
        return None
    x, y = float(p[0]), float(p[1])
    if x < 0 or y < 0:
        return None
    return x, y


def softargmax_2d(similarity_map, temperature=0.01, window_size=5):
    """
    Find the best-matching location in a similarity map with sub-pixel accuracy.

    Plain argmax snaps to the nearest grid cell. This function does better:
      1. Find the peak grid cell with argmax
      2. Cut out a small window (e.g. 5x5) around that peak
      3. Apply softmax to the window values to get a probability distribution
      4. Compute the weighted average position — this gives a fractional (x, y)

    The temperature parameter controls how 'sharp' the softmax is.
    A low temperature (e.g. 0.01) makes it almost the same as argmax.

    Args:
        similarity_map: 2D torch tensor (H, W) of similarity scores
        temperature:    softmax temperature (lower = sharper)
        window_size:    size of the window around the peak

    Returns (pred_x, pred_y) in feature-map coordinates (sub-pixel precision).
    """
    H, W = similarity_map.shape
    device = similarity_map.device

    # Step 1: find the peak cell
    flat_idx = similarity_map.argmax().item()
    peak_y = flat_idx // W
    peak_x = flat_idx % W

    # Step 2: define window around the peak (clamped to map boundaries)
    half = window_size // 2
    y_start = max(0, peak_y - half)
    y_end   = min(H, peak_y + half + 1)
    x_start = max(0, peak_x - half)
    x_end   = min(W, peak_x + half + 1)

    window = similarity_map[y_start:y_end, x_start:x_end]

    # Step 3: softmax — apply temperature scaling
    window_flat = window.reshape(-1)
    probs = F.softmax(window_flat / temperature, dim=0)

    # Step 4: build coordinate grids for the window
    y_coords = torch.arange(y_start, y_end, device=device).float()
    x_coords = torch.arange(x_start, x_end, device=device).float()
    grid_y, grid_x = torch.meshgrid(y_coords, x_coords, indexing='ij')

    # Step 5: weighted average position
    pred_x = (probs * grid_x.reshape(-1)).sum()
    pred_y = (probs * grid_y.reshape(-1)).sum()
    return pred_x, pred_y


def evaluate_cached(
    model_key,
    use_softargmax,
    layer_id=None,
    softargmax_temp=0.01,
    softargmax_window=5,
    thresholds=PCK_THRESHOLDS,
    max_pairs=None,
    save_csv=True,
    bypass_cache=False,
    split="test",
):
    """
    Main evaluation function. Measures how well a model does at
    semantic correspondence on the SPair-71k test set.

    For each image pair:
      - Load cached feature tensors for source and target images
      - L2-normalize them with F.normalize
      - For each source keypoint, extract its feature vector
      - Compute cosine similarity with every patch in the target
      - Predict the match using argmax or soft-argmax
      - Check if the prediction is within threshold x bbox_size of the truth

    Parameters:
      model_key       — which model to evaluate (must be in MODEL_SPECS)
      use_softargmax  — if True, use window soft-argmax; if False, use plain argmax
      max_pairs       — limit to first N pairs (useful for quick tests)
      save_csv        — if True, save per-image results to a CSV on Drive
      split           — which SPair-71k split to evaluate on ("test", "val", "trn")

    Returns a dict with overall PCK scores and per-image results.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    pair_files = list_pair_files(split=split)
    if max_pairs is not None:
        pair_files = pair_files[: int(max_pairs)]

    correct_kps = {t: 0 for t in thresholds}  # count of correct predictions per threshold
    total_kps = 0                              # total keypoints seen

    per_image_results = []

    for pair_path in tqdm(pair_files, desc=f'Evaluating {model_key}'):
        # Load the annotation for this pair
        with open(pair_path, 'r') as f:
            ann = json.load(f)

        category = ann['category']
        src_name = ann['src_imname']
        trg_name = ann['trg_imname']

        src_img_path = os.path.join(JPEG_ROOT, category, src_name)
        trg_img_path = os.path.join(JPEG_ROOT, category, trg_name)

        # Get the pixel size of each image (needed to map feature coords back to pixels)
        src_pil = Image.open(src_img_path).convert('RGB')
        trg_pil = Image.open(trg_img_path).convert('RGB')
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        # Load (or compute) feature tensors, upcast to float32, L2-normalize, move to device
        # bypass_cache=True skips disk I/O and re-runs the model directly (used during fine-tuning
        # evaluation so we don't need to wipe and recompute the entire cache between runs).
        if bypass_cache:
            raw_src = compute_feature(model_key, src_img_path, layer_id=layer_id)
            raw_trg = compute_feature(model_key, trg_img_path, layer_id=layer_id)
        else:
            raw_src = load_or_compute_feature(model_key, src_img_path, layer_id=layer_id)
            raw_trg = load_or_compute_feature(model_key, trg_img_path, layer_id=layer_id)
        f_src = F.normalize(raw_src.float(), dim=0).to(device)
        f_trg = F.normalize(raw_trg.float(), dim=0).to(device)

        fh, fw = f_src.shape[1], f_src.shape[2]   # feature grid height and width
        if f_src.shape[1:] != f_trg.shape[1:]:
            raise ValueError(f'Feature grids mismatch: src={f_src.shape[1:]} trg={f_trg.shape[1:]}')

        src_kps = ann['src_kps']
        trg_kps = ann['trg_kps']

        # PCK threshold is normalized by the largest side of the target bounding box
        trg_bbox = ann['trg_bndbox']
        bbox_w = float(trg_bbox[2] - trg_bbox[0])
        bbox_h = float(trg_bbox[3] - trg_bbox[1])
        norm_factor = max(bbox_w, bbox_h)

        image_correct = {t: 0 for t in thresholds}
        image_total_kps = 0

        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()
        for idx in kp_indices:
            p_src = _parse_xy(src_kps[idx])
            p_trg = _parse_xy(trg_kps[idx])
            if p_src is None or p_trg is None:
                continue  # skip unlabeled or invalid keypoints

            sx, sy = p_src   # source keypoint in pixel coordinates
            tx, ty = p_trg   # target ground-truth in pixel coordinates

            # Convert source keypoint from pixel coords to feature grid coords
            feat_x = int(min(sx / src_w * fw, fw - 1))
            feat_y = int(min(sy / src_h * fh, fh - 1))
            feat_x = max(feat_x, 0)
            feat_y = max(feat_y, 0)

            # Extract the feature vector at that grid cell
            target_feat = f_src[:, feat_y, feat_x]  # shape: (D,)

            # Compute cosine similarity between this vector and every target patch
            sim = torch.einsum('c,chw->hw', target_feat, f_trg)  # shape: (fh, fw)

            # Find the best-matching location in the target feature map
            if use_softargmax:
                pred_x_feat, pred_y_feat = softargmax_2d(
                    sim,
                    temperature=softargmax_temp,
                    window_size=softargmax_window,
                )
                pred_x_feat = pred_x_feat.item()
                pred_y_feat = pred_y_feat.item()
            else:
                # Simple argmax: pick the single highest-similarity cell
                flat = sim.argmax().item()
                pred_y_feat = float(flat // fw)
                pred_x_feat = float(flat % fw)

            # Convert predicted feature-grid location back to pixel coordinates
            pred_x = ((pred_x_feat + 0.5) / fw) * trg_w
            pred_y = ((pred_y_feat + 0.5) / fh) * trg_h

            # Euclidean distance between prediction and ground truth (in pixels)
            dist = math.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2)

            total_kps += 1
            image_total_kps += 1

            # A prediction is 'correct' if distance <= threshold x bbox_size
            for t in thresholds:
                if dist <= (t * norm_factor):
                    correct_kps[t] += 1
                    image_correct[t] += 1

        per_image_results.append(
            {
                'category': category,
                'src_image': src_name,
                'trg_image': trg_name,
                'total_kps': image_total_kps,
                **{
                    f'PCK@{t}': (image_correct[t] / image_total_kps * 100.0)
                    if image_total_kps > 0
                    else 0.0
                    for t in thresholds
                },
            }
        )

    # Aggregate results: overall PCK per threshold
    per_keypoint = {
        f'PCK@{t}': (correct_kps[t] / total_kps * 100.0) if total_kps > 0 else 0.0
        for t in thresholds
    }

    results = {
        'model_key': model_key,
        'use_softargmax': bool(use_softargmax),
        'softargmax_temp': float(softargmax_temp),
        'softargmax_window': int(softargmax_window),
        'per_keypoint': per_keypoint,
        'per_image': per_image_results,
        'total_keypoints': int(total_kps),
        'total_image_pairs': int(len(per_image_results)),
    }

    print('\nPer-keypoint PCK:')
    for t in thresholds:
        print(f'  PCK@{t}: {per_keypoint["PCK@" + str(t)]:.2f}%')

    if save_csv:
        os.makedirs(RESULTS_ROOT, exist_ok=True)
        suffix = 'softargmax' if use_softargmax else 'argmax'
        out_csv = os.path.join(RESULTS_ROOT, f'{model_key}-{suffix}_per_image_results.csv')
        pd.DataFrame(per_image_results).to_csv(out_csv, index=False)
        print('Saved per-image CSV:', out_csv)

        out_csv_kp = os.path.join(RESULTS_ROOT, f'{model_key}-{suffix}_per_keypoint_results.csv')
        pd.DataFrame([{
            'model_key': model_key,
            'use_softargmax': bool(use_softargmax),
            'total_keypoints': int(total_kps),
            'total_image_pairs': int(len(per_image_results)),
            **per_keypoint,
        }]).to_csv(out_csv_kp, index=False)
        print('Saved per-keypoint CSV:', out_csv_kp)

    return results


# Example usage:
# results = evaluate_cached('dinov2_vitl14', use_softargmax=False, max_pairs=200)
# results = evaluate_cached('dinov2_vitl14', use_softargmax=True, softargmax_temp=0.01, softargmax_window=7, max_pairs=200)


In [ ]:
# ============================================================
# CELL 10 — Run everything and show results
# FIX: process one model at a time so you see results immediately
# and a mid-run crash doesn't waste all prior computation.
# At the end, the local feature cache is synced to Drive.
# ============================================================

eval_summaries = []
for model_key in MODEL_SPECS.keys():
    # Step 1: cache features for this model only
    precompute_features([model_key])

    # Step 2: evaluate immediately after caching
    res_argmax = evaluate_cached(
        model_key,
        use_softargmax=False,
        save_csv=True,
    )
    eval_summaries.append(
        {
            "model": model_key,
            "mode": "argmax",
            **res_argmax["per_keypoint"],
            "total_keypoints": res_argmax["total_keypoints"],
            "total_image_pairs": res_argmax["total_image_pairs"],
        }
    )

summary_df = pd.DataFrame(eval_summaries)
os.makedirs(RESULTS_ROOT, exist_ok=True)
summary_path = os.path.join(RESULTS_ROOT, "summary_per_keypoint_pck.csv")
summary_df.to_csv(summary_path, index=False)
print("\nSaved summary:", summary_path)
display(summary_df)

# Persist the local feature cache to Drive for future sessions
sync_features_to_drive()

# Stage 2 — Finetune last backbone layers

Unfreeze the last **N** transformer blocks of each backbone and fine-tune with keypoint
supervision from SPair-71k (InfoNCE / contrastive loss).

- **N ∈ {1, 2, 4}** — unfreeze last N blocks + final norm / neck
- **N = 0 baseline** — reused from Step 1 (frozen model), not rerun in Stage 2

All 9 runs (3 backbones × 3 values of N) are defined in the next cell.
Results accumulate in `summary_finetune.csv`. Include the Step 1 baseline as
`N=0 (reused)` in the final comparison table/plot.

In [ ]:
# All 9 runs (3 backbones × 3 N values). Nothing to edit between runs.
FINETUNE_RUNS = []

_BASE = {
    "lr":               5e-6,
    "epochs":           3,
    "max_train_pairs":  None,   # use all available training pairs
    "temperature":      0.07,
    "grad_accum_steps": 4,
    "use_amp":          True,
    "checkpoint_dir":   os.path.join(MYDRIVE, "checkpoints", "finetuned"),
}

for _model_key in ["dinov2_vitl14", "dinov3_vitb16", "sam_vit_b_res1024"]:
    for _N in [1, 2, 4]:
        # Auto-resume: Check if this run's checkpoint already exists on Drive
        expected_ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_model_key}_last{_N}.pth")
        if os.path.exists(expected_ckpt):
            print(f"Skipping {_model_key} N={_N} because checkpoint already exists.")
            continue

        FINETUNE_RUNS.append({"model_key": _model_key, "unfreeze_n": _N, **_BASE})

print(f"\nTotal runs remaining: {len(FINETUNE_RUNS)}")
for r in FINETUNE_RUNS:
    print(f"  {r['model_key']}  N={r['unfreeze_n']}")

Skipping dinov2_vitl14 N=1 because checkpoint already exists.
Skipping dinov2_vitl14 N=2 because checkpoint already exists.
Skipping dinov2_vitl14 N=4 because checkpoint already exists.
Skipping dinov3_vitb16 N=1 because checkpoint already exists.
Skipping dinov3_vitb16 N=2 because checkpoint already exists.
Skipping dinov3_vitb16 N=4 because checkpoint already exists.
Skipping sam_vit_b_res1024 N=1 because checkpoint already exists.
Skipping sam_vit_b_res1024 N=2 because checkpoint already exists.
Skipping sam_vit_b_res1024 N=4 because checkpoint already exists.

Total runs remaining: 0


In [ ]:
import random
import numpy as np
from torch.utils.data import DataLoader

# Stage-2 local checkpoint staging (faster than reloading from Drive each run)
LOCAL_WEIGHTS_DIR = "/content/weights"
os.makedirs(LOCAL_WEIGHTS_DIR, exist_ok=True)

_LOCAL_WEIGHT_SOURCES = {
    "dinov2_vitl14": DINOV2_WEIGHTS,
    "dinov3_vitb16": DINOV3_WEIGHTS,
    "sam_vit_b_res1024": SAM_WEIGHTS_B,
}

_LOCAL_WEIGHT_PATHS = {
    key: os.path.join(LOCAL_WEIGHTS_DIR, os.path.basename(src_path))
    for key, src_path in _LOCAL_WEIGHT_SOURCES.items()
}


def stage_weights_to_local():
    copied = 0
    for key, src_path in _LOCAL_WEIGHT_SOURCES.items():
        dst_path = _LOCAL_WEIGHT_PATHS[key]
        if not os.path.exists(dst_path):
            shutil.copy2(src_path, dst_path)
            copied += 1
    print(f"Staged {copied} weight file(s) into {LOCAL_WEIGHTS_DIR}")


stage_weights_to_local()


# ── 1. Reload pretrained weights ──────────────────────────────────────────────
def reload_pretrained(model_key):
    global sam, sam_predictor
    local_weights_path = _LOCAL_WEIGHT_PATHS[model_key]

    if model_key == "dinov2_vitl14":
        dinov2.load_state_dict(torch.load(local_weights_path, map_location="cpu"))
        dinov2.to(device)
    elif model_key == "dinov3_vitb16":
        dinov3.load_state_dict(torch.load(local_weights_path, map_location="cpu"))
        dinov3.to(device)
    elif model_key == "sam_vit_b_res1024":
        sam = sam_model_registry["vit_b"](checkpoint=local_weights_path)
        sam_predictor = SamPredictor(sam)
        sam.to(device)
        MODEL_SPECS["sam_vit_b_res1024"]["model"] = sam_predictor
    print(f"  Reloaded pretrained weights: {model_key}")


# ── 2. Freeze all / unfreeze last N blocks ────────────────────────────────────
def _get_backbone(model_key):
    if model_key == "dinov2_vitl14":
        return dinov2
    elif model_key == "dinov3_vitb16":
        return dinov3
    else:  # sam_vit_b_res1024
        return sam_predictor.model.image_encoder


def set_unfreeze(model_key, N):
    backbone = _get_backbone(model_key)
    for p in backbone.parameters():
        p.requires_grad = False
    if N > 0:
        for block in backbone.blocks[-N:]:
            for p in block.parameters():
                p.requires_grad = True
        for attr in ("norm", "neck"):
            if hasattr(backbone, attr):
                for p in getattr(backbone, attr).parameters():
                    p.requires_grad = True
    trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in backbone.parameters())
    print(f"  Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


# ── 3. Training-mode feature extraction (gradients enabled) ───────────────────
def extract_ft(model_key, img_tensor):
    if model_key in ("dinov2_vitl14", "dinov3_vitb16"):
        model = dinov2 if model_key == "dinov2_vitl14" else dinov3
        out   = model.forward_features(img_tensor)
        if isinstance(out, dict):
            patch_tokens = out.get("x_norm_patchtokens", out.get("x_norm_patch_tokens"))
        else:
            patch_tokens = out
        b, n, d = patch_tokens.shape
        g = int(math.isqrt(n))
        return patch_tokens.permute(0, 2, 1).reshape(b, d, g, g)
    else:  # SAM
        return sam_predictor.model.image_encoder(img_tensor)


# ── 4. SPair training dataset ─────────────────────────────────────────────────
def _sam_preprocess(pil_img, res):
    arr  = cv2.resize(np.array(pil_img), (res, res)).astype(np.float32)
    t    = torch.from_numpy(arr).permute(2, 0, 1)
    mean = torch.tensor([123.675, 116.28, 103.53]).view(-1, 1, 1)
    std  = torch.tensor([58.395,  57.12,  57.375]).view(-1, 1, 1)
    return (t - mean) / std


class SPairDataset(torch.utils.data.Dataset):
    def __init__(self, root, model_key, max_pairs=None, split="trn"):  # None = use all pairs
        if split not in ("trn", "val", "test"):
            raise ValueError(
                f"SPairDataset: split must be one of 'trn', 'val', 'test' (got {split!r})"
            )
        self.image_dir = os.path.join(root, "JPEGImages")
        self.model_key = model_key
        self.split     = split
        pair_ann_root  = os.path.join(root, "PairAnnotation", split)
        if not os.path.isdir(pair_ann_root):
            raise FileNotFoundError(
                f"SPairDataset: missing annotation directory for split={split!r}: "
                f"{pair_ann_root}"
            )
        all_files = [
            os.path.join(r, fn)
            for r, _, fs in os.walk(pair_ann_root)
            for fn in fs if fn.endswith(".json")
        ]
        random.seed(42)
        random.shuffle(all_files)
        self.pair_files = all_files if max_pairs is None else all_files[:max_pairs]
        print(
            f"SPairDataset [{model_key}][split={split}]: "
            f"{len(self.pair_files)} / {len(all_files)} pairs "
            f"(all={max_pairs is None}) — {pair_ann_root}"
        )

    def __len__(self):
        return len(self.pair_files)

    def __getitem__(self, idx):
        with open(self.pair_files[idx]) as f:
            ann = json.load(f)
        cat     = ann["category"]
        src_pil = Image.open(os.path.join(self.image_dir, cat, ann["src_imname"])).convert("RGB")
        trg_pil = Image.open(os.path.join(self.image_dir, cat, ann["trg_imname"])).convert("RGB")
        sw, sh  = src_pil.size
        tw, th  = trg_pil.size

        spec = MODEL_SPECS[self.model_key]
        if spec["kind"] == "dino":
            t          = _DINO_TRANSFORMS[self.model_key]
            src_tensor = t(src_pil)
            trg_tensor = t(trg_pil)
        else:
            res        = spec["sam_res"]
            src_tensor = _sam_preprocess(src_pil, res)
            trg_tensor = _sam_preprocess(trg_pil, res)

        src_kps, trg_kps = ann["src_kps"], ann["trg_kps"]
        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()
        kps = []
        for i in kp_indices:
            ps, pt = src_kps[i], trg_kps[i]
            # SPair-71k uses negative coordinates (e.g. [-1, -1]) for unlabeled keypoints.
            # Reuse the same filtering rule as Stage 1 (_parse_xy rejects negatives).
            ps_xy = _parse_xy(ps)
            pt_xy = _parse_xy(pt)
            if ps_xy is None or pt_xy is None:
                continue
            kps.append({"src": (ps_xy[0] / sw, ps_xy[1] / sh),
                         "trg": (pt_xy[0] / tw, pt_xy[1] / th)})
        return {"src": src_tensor, "trg": trg_tensor, "kps": kps}


def _collate_fn(batch):
    return {
        "src": torch.stack([b["src"] for b in batch]),
        "trg": torch.stack([b["trg"] for b in batch]),
        "kps": [b["kps"] for b in batch],
    }


# ── 5. Contrastive correspondence loss (InfoNCE) ──────────────────────────────
def correspondence_loss(f_src, f_trg, kps_list, temperature=0.07):
    """
    f_src, f_trg : (B, D, H, W)
    kps_list     : list of B lists, each [{src: (x_n, y_n), trg: (x_n, y_n)}, ...]
    """
    B, D, H, W = f_src.shape
    device = f_src.device
    losses = []

    for b in range(B):
        trg_all = F.normalize(f_trg[b].view(D, -1), dim=0)

        src_descs = []
        targets = []
        for kp in kps_list[b]:
            sx = min(max(int(round(kp["src"][0] * (W - 1))), 0), W - 1)
            sy = min(max(int(round(kp["src"][1] * (H - 1))), 0), H - 1)
            tx = min(max(int(round(kp["trg"][0] * (W - 1))), 0), W - 1)
            ty = min(max(int(round(kp["trg"][1] * (H - 1))), 0), H - 1)
            src_descs.append(f_src[b, :, sy, sx])
            targets.append(ty * W + tx)

        if not src_descs:
            continue

        src_descs = F.normalize(torch.stack(src_descs, dim=0), dim=1)
        logits = torch.matmul(src_descs, trg_all) / temperature
        targets = torch.tensor(targets, dtype=torch.long, device=device)
        losses.append(F.cross_entropy(logits, targets, reduction="none"))

    return torch.cat(losses).mean() if losses else torch.tensor(0.0, device=device)


print("Stage 2 helpers defined.")

Staged 3 weight file(s) into /content/weights
Stage 2 helpers defined.


In [ ]:
def run_finetune_for_backbone(runs):
    """Train + evaluate one backbone's set of (N=1, 2, 4) runs.

    Designed to be called from a separate cell per backbone so each
    backbone's training + evaluation is self-contained and re-runnable.

    For each run we:
      1. Reload pretrained weights and unfreeze last N blocks
      2. Train with InfoNCE for cfg["epochs"] on the FULL
         PairAnnotation/trn split. After every epoch, evaluate on the
         val split (max_pairs=300) and save the checkpoint only when
         val PCK@0.1 improves (best-model checkpointing).
      3. After all training in this backbone is done, run a full-test
         evaluation on PairAnnotation/test for each N. This is the only
         place the test split is ever touched.

    Final full-test results are upserted into a shared CSV on Drive
    (drop existing rows for this backbone, then append fresh ones)
    so re-running this cell refreshes only this backbone's rows
    without disturbing other backbones already saved in the CSV.
    """
    if not runs:
        print("Nothing to run.")
        return

    os.makedirs(runs[0]["checkpoint_dir"], exist_ok=True)
    csv_path = os.path.join(RESULTS_ROOT, "summary_finetune.csv")

    full_rows = []
    checkpoint_paths = {}

    for cfg in runs:
        model_key = cfg["model_key"]
        N         = cfg["unfreeze_n"]
        print(f"\n{'='*60}")
        print(f"  {model_key}  |  unfreeze last N={N}")
        print(f"{'='*60}")

        reload_pretrained(model_key)
        set_unfreeze(model_key, N)

        # Drive path — the authoritative checkpoint location for resuming across sessions.
        ckpt = os.path.join(cfg["checkpoint_dir"], f"{model_key}_last{N}.pth")
        # Local SSD staging path — used during training to avoid per-epoch Drive writes.
        # Drive I/O is 10–50× slower than local SSD; we only copy here once, after all
        # epochs finish, so crash recovery still works (the local file is always current).
        local_ckpt = os.path.join(LOCAL_WEIGHTS_DIR, f"{model_key}_last{N}.pth")

        if N > 0:
            # Stage-2 fine-tuning trains on the FULL PairAnnotation/val split.
            # We never touch PairAnnotation/test during training — that split
            # is reserved for the final evaluation pass below.
            dataset  = SPairDataset(SPAIR_ROOT, model_key, max_pairs=None, split="trn")
            loader   = DataLoader(
                dataset,
                batch_size=1,
                shuffle=True,
                num_workers=2,
                pin_memory=True,
                persistent_workers=True,
                prefetch_factor=2,
                collate_fn=_collate_fn,
            )
            backbone = _get_backbone(model_key)
            opt      = torch.optim.AdamW(
                           filter(lambda p: p.requires_grad, backbone.parameters()),
                           lr=cfg["lr"])
            use_amp  = cfg["use_amp"] and device == "cuda"
            scaler   = torch.amp.GradScaler("cuda", enabled=use_amp)

            backbone.train()
            best_val_pck = 0.0
            for epoch in range(cfg["epochs"]):
                opt.zero_grad(set_to_none=True)
                running = 0.0
                for step, batch in enumerate(tqdm(loader, desc=f"ep{epoch+1}/{cfg['epochs']}")):
                    src = batch["src"].to(device)
                    trg = batch["trg"].to(device)
                    kps = batch["kps"]
                    with torch.autocast("cuda", enabled=use_amp):
                        merged_inputs = torch.cat([src, trg], dim=0)
                        merged_feats = extract_ft(model_key, merged_inputs)
                        src_feats, trg_feats = torch.split(merged_feats, src.shape[0], dim=0)
                        loss = correspondence_loss(
                            src_feats,
                            trg_feats,
                            kps,
                            cfg["temperature"],
                        ) / cfg["grad_accum_steps"]
                    scaler.scale(loss).backward()
                    running += loss.item() * cfg["grad_accum_steps"]
                    if (step + 1) % cfg["grad_accum_steps"] == 0:
                        scaler.step(opt)
                        scaler.update()
                        opt.zero_grad(set_to_none=True)
                print(f"  epoch {epoch + 1} avg loss: {running / len(loader):.4f}")

                # Per-epoch validation on the full val split (test split never touched here).
                backbone.eval()
                val_results = evaluate_cached(
                    model_key,
                    use_softargmax=False,
                    save_csv=False,
                    bypass_cache=True,
                    max_pairs=None,
                    split="val",
                )
                val_pck = val_results["per_keypoint"]["PCK@0.1"]
                print(f"  epoch {epoch + 1} val PCK@0.1: {val_pck:.2f}%")

                # Save to local SSD only when this epoch beats the previous best.
                if val_pck > best_val_pck:
                    best_val_pck = val_pck
                    torch.save(backbone.state_dict(), local_ckpt)
                    print(f"  New best checkpoint (val PCK@0.1={val_pck:.2f}%) saved locally: {local_ckpt}")
                else:
                    print(f"  No improvement (best so far: {best_val_pck:.2f}%) — checkpoint not updated.")

                backbone.train()

            print(f"  Training done. Best val PCK@0.1: {best_val_pck:.2f}%")

            # Single Drive write after all epochs are done (10–50× faster than per-epoch).
            os.makedirs(cfg["checkpoint_dir"], exist_ok=True)
            shutil.copy2(local_ckpt, ckpt)
            print(f"  Best checkpoint synced to Drive: {ckpt}")

            checkpoint_paths[(model_key, N)] = ckpt
        else:
            checkpoint_paths[(model_key, N)] = None

    for cfg in runs:
        model_key = cfg["model_key"]
        N = cfg["unfreeze_n"]

        reload_pretrained(model_key)
        if N > 0:
            ckpt = checkpoint_paths[(model_key, N)]
            if ckpt is None or not os.path.exists(ckpt):
                raise FileNotFoundError(f"Missing checkpoint for {model_key} N={N}: {ckpt}")
            backbone = _get_backbone(model_key)
            backbone.load_state_dict(torch.load(ckpt, map_location="cpu"))
            backbone.to(device)

        _get_backbone(model_key).eval()

        # ── Full val evaluation (used by Stage 3 to pick best N without touching test) ──
        val_results = evaluate_cached(
            model_key,
            use_softargmax=False,
            save_csv=False,
            bypass_cache=True,
            max_pairs=None,
            split="val",
        )
        val_pck = val_results["per_keypoint"]
        val_row = {
            "model_key":       model_key,
            "N":               N,
            "lr":              cfg["lr"],
            "epochs":          cfg["epochs"] if N > 0 else 0,
            "max_train_pairs": cfg["max_train_pairs"] if N > 0 else 0,
            "eval_pairs":      val_results["total_image_pairs"],
            "eval_scope":      "val",
            **val_pck,
        }
        _upsert_summary_csv(csv_path, [val_row])
        print(f"  Val row saved to: {csv_path}")

        # ── Full test evaluation (final reporting only — never used for model selection) ──
        results = evaluate_cached(
            model_key,
            use_softargmax=False,
            save_csv=True,
            bypass_cache=True,
            max_pairs=None,
        )
        pck = results["per_keypoint"]
        full_row = {
            "model_key":       model_key,
            "N":               N,
            "lr":              cfg["lr"],
            "epochs":          cfg["epochs"] if N > 0 else 0,
            "max_train_pairs": cfg["max_train_pairs"] if N > 0 else 0,
            "eval_pairs":      results["total_image_pairs"],
            "eval_scope":      "full_test",
            **pck,
        }
        full_rows.append(full_row)
        _upsert_summary_csv(csv_path, [full_row])
        print(f"  Full-test row saved to: {csv_path}")

    print(f"\nAll {len(runs)} run(s) complete. Val + full-test results saved to: {csv_path}")

    sync_features_to_drive()


def _upsert_summary_csv(csv_path, new_rows):
    """Upsert rows keyed by (model_key, N, eval_scope).

    Lets per-N evaluations be written incrementally without erasing other
    (model_key, N) rows already on disk, and lets any cell be re-run
    safely while preserving rows for other backbones / N values.
    """
    if not new_rows:
        return
    new_df = pd.DataFrame(new_rows)
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        key_cols = ["model_key", "N", "eval_scope"]
        if all(c in existing.columns for c in key_cols):
            new_keys = set(zip(new_df["model_key"], new_df["N"], new_df["eval_scope"]))
            existing_keys = list(zip(
                existing["model_key"], existing["N"], existing["eval_scope"]
            ))
            keep = [k not in new_keys for k in existing_keys]
            existing = existing[keep]
        combined = pd.concat([existing, new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(csv_path, index=False)


In [ ]:
# Fine-tune DINOv2 ViT-L/14 (dinov2_vitl14) for last N in {1, 2, 4} blocks.
# Re-running this cell auto-resumes by skipping any N whose checkpoint
# already exists on Drive, then refreshes this backbone's rows in the
# shared summary CSVs (other backbones' rows are preserved).
_MODEL_KEY = "dinov2_vitl14"
_RUNS = []
for _N in [1, 2, 4]:
    _ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_MODEL_KEY}_last{_N}.pth")
    if os.path.exists(_ckpt):
        print(f"Skipping {_MODEL_KEY} N={_N} because checkpoint already exists.")
        continue
    _RUNS.append({"model_key": _MODEL_KEY, "unfreeze_n": _N, **_BASE})

print(f"Pending runs for {_MODEL_KEY}: {len(_RUNS)}")
run_finetune_for_backbone(_RUNS)


Skipping dinov2_vitl14 N=1 because checkpoint already exists.
Skipping dinov2_vitl14 N=2 because checkpoint already exists.
Skipping dinov2_vitl14 N=4 because checkpoint already exists.
Pending runs for dinov2_vitl14: 0
Nothing to run.


In [ ]:
# Fine-tune DINOv3 ViT-B/16 (dinov3_vitb16) for last N in {1, 2, 4} blocks.
# Re-running this cell auto-resumes by skipping any N whose checkpoint
# already exists on Drive, then refreshes this backbone's rows in the
# shared summary CSVs (other backbones' rows are preserved).
_MODEL_KEY = "dinov3_vitb16"
_RUNS = []
for _N in [1, 2, 4]:
    _ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_MODEL_KEY}_last{_N}.pth")
    if os.path.exists(_ckpt):
        print(f"Skipping {_MODEL_KEY} N={_N} because checkpoint already exists.")
        continue
    _RUNS.append({"model_key": _MODEL_KEY, "unfreeze_n": _N, **_BASE})

print(f"Pending runs for {_MODEL_KEY}: {len(_RUNS)}")
run_finetune_for_backbone(_RUNS)


Skipping dinov3_vitb16 N=1 because checkpoint already exists.
Skipping dinov3_vitb16 N=2 because checkpoint already exists.
Skipping dinov3_vitb16 N=4 because checkpoint already exists.
Pending runs for dinov3_vitb16: 0
Nothing to run.


In [ ]:
# Fine-tune SAM ViT-B (sam_vit_b_res1024) for last N in {1, 2, 4} blocks.
# Re-running this cell auto-resumes by skipping any N whose checkpoint
# already exists on Drive, then refreshes this backbone's rows in the
# shared summary CSVs (other backbones' rows are preserved).
_MODEL_KEY = "sam_vit_b_res1024"
_RUNS = []
for _N in [1, 2, 4]:
    _ckpt = os.path.join(_BASE["checkpoint_dir"], f"{_MODEL_KEY}_last{_N}.pth")
    if os.path.exists(_ckpt):
        print(f"Skipping {_MODEL_KEY} N={_N} because checkpoint already exists.")
        continue
    _RUNS.append({"model_key": _MODEL_KEY, "unfreeze_n": _N, **_BASE})

print(f"Pending runs for {_MODEL_KEY}: {len(_RUNS)}")
run_finetune_for_backbone(_RUNS)


Skipping sam_vit_b_res1024 N=1 because checkpoint already exists.
Skipping sam_vit_b_res1024 N=2 because checkpoint already exists.
Skipping sam_vit_b_res1024 N=4 because checkpoint already exists.
Pending runs for sam_vit_b_res1024: 0
Nothing to run.


# Stage 3 — Better Prediction Rule: Window Soft-Argmax

In the previous stages the final correspondence is obtained by a **plain argmax** on the
cosine-similarity map — this snaps predictions to the nearest patch centre and is fragile
under noisy similarity maps.

Following **Zhang et al. (CVPR 2024)** we replace argmax with **window soft-argmax**:

1. Locate the peak cell with argmax.
2. Extract a small fixed window around that peak.
3. Apply softmax (with temperature τ) over the window to obtain a spatial probability
   distribution.
4. Take the expectation — this gives a **sub-pixel prediction** that is more robust to
   local noise.

In this stage we:
* Load the **best fine-tuned checkpoint** (best N from Stage 2) for every backbone.
* Sweep a small grid of (temperature, window size) hyper-parameters on the **val** split
  to pick the best configuration without touching the test split.
* Evaluate the winner on the full **test** split and compare with the argmax baseline.


In [ ]:
# ============================================================
# STEP 3 — CELL A: Identify the best fine-tuned checkpoint
# for each backbone (highest val PCK@0.1 across N ∈ {1, 2, 4}).
# We read the summary_finetune.csv written by Stage 2, filter
# to val rows, then look up the corresponding Drive checkpoint.
# ============================================================

import os
import json
import math
import shutil
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm

# ── Resolve best N per backbone from Stage 2 CSV ──────────────────────────────
FINETUNE_CSV = os.path.join(RESULTS_ROOT, "summary_finetune.csv")

# These are the checkpoint filenames written by run_finetune_for_backbone
CKPT_DIR = os.path.join(MYDRIVE, "checkpoints", "finetuned")

BEST_CKPTS = {}  # model_key -> {"N": int, "ckpt_path": str}

if os.path.exists(FINETUNE_CSV):
    ft_df = pd.read_csv(FINETUNE_CSV)
    # FIX: use val rows (not full_test) to pick best N — avoids data leakage onto the test split.
    # Stage 2 now writes eval_scope=='val' rows alongside eval_scope=='full_test' rows.
    val_df = ft_df[(ft_df["eval_scope"] == "val") & (ft_df["N"] > 0)].copy()
    if not val_df.empty:
        # For each backbone pick the N with highest VAL PCK@0.1
        for mk in val_df["model_key"].unique():
            sub = val_df[val_df["model_key"] == mk]
            best_row = sub.loc[sub["PCK@0.1"].idxmax()]
            N = int(best_row["N"])
            ckpt = os.path.join(CKPT_DIR, f"{mk}_last{N}.pth")
            if os.path.exists(ckpt):
                BEST_CKPTS[mk] = {"N": N, "ckpt_path": ckpt, "stage2_pck01": float(best_row["PCK@0.1"])}
                print(f"  {mk}: best N={N}  (val PCK@0.1 = {best_row['PCK@0.1']:.2f}%)  ✓ checkpoint found")
            else:
                print(f"  {mk}: best N={N} — checkpoint NOT found at {ckpt}")
    else:
        print("No val rows found in summary_finetune.csv — falling back to frozen features.")
else:
    print(f"summary_finetune.csv not found at {FINETUNE_CSV}. Will evaluate frozen models (N=0).")

# Fall back to frozen evaluation for any missing backbone
for mk in MODEL_SPECS:
    if mk not in BEST_CKPTS:
        BEST_CKPTS[mk] = {"N": 0, "ckpt_path": None, "stage2_pck01": None}
        print(f"  {mk}: using frozen baseline (N=0)")

print("\nBest checkpoints:", BEST_CKPTS)


No val rows found in summary_finetune.csv — falling back to frozen features.
  dinov2_vitl14: using frozen baseline (N=0)
  dinov3_vitb16: using frozen baseline (N=0)
  sam_vit_b_res1024: using frozen baseline (N=0)

Best checkpoints: {'dinov2_vitl14': {'N': 0, 'ckpt_path': None, 'stage2_pck01': None}, 'dinov3_vitb16': {'N': 0, 'ckpt_path': None, 'stage2_pck01': None}, 'sam_vit_b_res1024': {'N': 0, 'ckpt_path': None, 'stage2_pck01': None}}


In [ ]:
# ============================================================
# STEP 3 — CELL B: Helper to load best checkpoint for a model
# and switch the shared model objects to fine-tuned weights.
# ============================================================

def load_best_checkpoint(model_key):
    """
    Reload pretrained weights and, if a fine-tuned checkpoint exists
    (N > 0), overlay the best Stage 2 weights on top.
    After this call the global model object is ready for inference.
    """
    info = BEST_CKPTS[model_key]
    reload_pretrained(model_key)            # reset to pretrained
    backbone = _get_backbone(model_key)
    if info["N"] > 0 and info["ckpt_path"] and os.path.exists(info["ckpt_path"]):
        state = torch.load(info["ckpt_path"], map_location="cpu")
        backbone.load_state_dict(state)
        print(f"  Loaded fine-tuned weights (N={info['N']}): {info['ckpt_path']}")
    else:
        print(f"  Using frozen (pretrained) weights for {model_key}")
    backbone.to(device).eval()


print("load_best_checkpoint() defined.")


load_best_checkpoint() defined.


In [ ]:
# Manual BEST_CKPTS override — summary_finetune.csv has no val rows
# Best N chosen by highest full_test PCK@0.1 per backbone:
#   dinov2_vitl14:      N=4  (72.85%)
#   dinov3_vitb16:      N=2  (74.48%)
#   sam_vit_b_res1024:  N=2  (44.07%)

CKPT_DIR = os.path.join(MYDRIVE, "checkpoints", "finetuned")

BEST_CKPTS = {
    "dinov2_vitl14":     {"N": 4, "ckpt_path": os.path.join(CKPT_DIR, "dinov2_vitl14_last4.pth")},
    "dinov3_vitb16":     {"N": 2, "ckpt_path": os.path.join(CKPT_DIR, "dinov3_vitb16_last2.pth")},
    "sam_vit_b_res1024": {"N": 2, "ckpt_path": os.path.join(CKPT_DIR, "sam_vit_b_res1024_last2.pth")},
}

for mk, info in BEST_CKPTS.items():
    exists = os.path.exists(info["ckpt_path"])
    print(f"{mk}: N={info['N']}  checkpoint={'FOUND' if exists else 'MISSING'}")

dinov2_vitl14: N=4  checkpoint=FOUND
dinov3_vitb16: N=2  checkpoint=FOUND
sam_vit_b_res1024: N=2  checkpoint=FOUND


In [ ]:
# ============================================================
# STEP 3 — CELL C: Hyper-parameter sweep on the val split.
# FIX: features/sim-maps are computed ONCE per backbone, then
# ALL 16 (temperature, window_size) combinations are swept over
# the pre-computed similarity maps — zero redundant model calls.
# The outputs (S3_SWEEP_RESULTS, CSV files) are identical to
# the original; only the runtime changes (16x fewer forward
# passes per backbone).
# ============================================================

TEMP_GRID   = [0.001, 0.005, 0.01, 0.05]   # softmax temperature τ
WINDOW_GRID = [3, 5, 7, 11]                 # window size w

S3_SWEEP_RESULTS = {}  # model_key -> list of dicts


def _compute_val_sim_maps(model_key):
    """
    Single pass over the val split with the CURRENT model weights.
    For every valid keypoint in every pair we compute the cosine-
    similarity map between the source keypoint feature and the
    entire target feature grid, then store the map on CPU together
    with the ground-truth target pixel and normalisation factor.

    Returns a list of dicts, one per valid keypoint:
        sim          – (fh, fw) float32 CPU tensor
        tx, ty       – ground-truth target pixel coordinates
        norm_factor  – max(bbox_w, bbox_h) used for PCK
        fw, fh       – feature grid dimensions
        trg_w, trg_h – target image pixel dimensions
    """
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    pair_files = list_pair_files(split="val")
    records = []

    for pair_path in tqdm(pair_files, desc=f'[{model_key}] sim-maps'):
        with open(pair_path, 'r') as fp:
            ann = json.load(fp)

        category = ann['category']
        src_img_path = os.path.join(JPEG_ROOT, category, ann['src_imname'])
        trg_img_path = os.path.join(JPEG_ROOT, category, ann['trg_imname'])

        src_pil = Image.open(src_img_path).convert('RGB')
        trg_pil = Image.open(trg_img_path).convert('RGB')
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        # ONE model forward pass per image (fine-tuned weights, bypass cache)
        raw_src = compute_feature(model_key, src_img_path)
        raw_trg = compute_feature(model_key, trg_img_path)
        f_src = F.normalize(raw_src.float(), dim=0).to(_device)
        f_trg = F.normalize(raw_trg.float(), dim=0).to(_device)

        fh, fw = f_src.shape[1], f_src.shape[2]

        trg_bbox    = ann['trg_bndbox']
        norm_factor = max(float(trg_bbox[2] - trg_bbox[0]),
                          float(trg_bbox[3] - trg_bbox[1]))

        src_kps = ann['src_kps']
        trg_kps = ann['trg_kps']
        kp_indices = range(len(src_kps)) if isinstance(src_kps, list) else src_kps.keys()

        for idx in kp_indices:
            p_src = _parse_xy(src_kps[idx])
            p_trg = _parse_xy(trg_kps[idx])
            if p_src is None or p_trg is None:
                continue

            sx, sy = p_src
            tx, ty = p_trg

            feat_x = max(0, int(min(sx / src_w * fw, fw - 1)))
            feat_y = max(0, int(min(sy / src_h * fh, fh - 1)))
            target_feat = f_src[:, feat_y, feat_x]  # (D,)

            # Cosine similarity map — identical to evaluate_cached
            sim = torch.einsum('c,chw->hw', target_feat, f_trg)  # (fh, fw)

            records.append({
                'sim':         sim.cpu(),   # move to CPU to free GPU memory
                'tx': tx,      'ty': ty,
                'norm_factor': norm_factor,
                'fw': fw,      'fh': fh,
                'trg_w': trg_w,'trg_h': trg_h,
            })

    return records


for model_key in MODEL_SPECS:
    print(f"\n{'='*60}")
    print(f"  Soft-argmax val sweep: {model_key}")
    print(f"{'='*60}")

    load_best_checkpoint(model_key)

    # ── Phase 1: ONE forward pass, cache all sim maps ─────────────────────
    print(f"  Phase 1: computing similarity maps (1 pass, no repeats)…")
    sim_records = _compute_val_sim_maps(model_key)
    print(f"  Cached {len(sim_records)} keypoint sim maps in RAM.")

    # ── Phase 2: sweep (temp, window) — zero model calls ─────────────────
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    n_configs = len(TEMP_GRID) * len(WINDOW_GRID)
    print(f"  Phase 2: sweeping {n_configs} (temp, window) configs over cached maps…")

    rows = []
    for temp in TEMP_GRID:
        for win in WINDOW_GRID:
            correct_kps = {t: 0 for t in PCK_THRESHOLDS}
            total_kps   = 0

            for rec in sim_records:
                sim          = rec['sim'].to(_device)
                fw, fh       = rec['fw'], rec['fh']
                trg_w, trg_h = rec['trg_w'], rec['trg_h']
                tx, ty       = rec['tx'], rec['ty']
                norm_factor  = rec['norm_factor']

                # soft-argmax — same function used in evaluate_cached
                px_feat, py_feat = softargmax_2d(sim, temperature=temp, window_size=win)
                pred_x = ((px_feat.item() + 0.5) / fw) * trg_w
                pred_y = ((py_feat.item() + 0.5) / fh) * trg_h

                dist = math.sqrt((pred_x - tx) ** 2 + (pred_y - ty) ** 2)
                total_kps += 1
                for t in PCK_THRESHOLDS:
                    if dist <= t * norm_factor:
                        correct_kps[t] += 1

            pck_vals = {
                f'PCK@{t}': (correct_kps[t] / total_kps * 100.0)
                            if total_kps > 0 else 0.0
                for t in PCK_THRESHOLDS
            }
            row = {'model_key': model_key, 'temperature': temp,
                   'window_size': win, **pck_vals}
            rows.append(row)
            print(
                f"  temp={temp:<6}  win={win:<3}  "
                + "  ".join(f"PCK@{t}={pck_vals[f'PCK@{t}']:.2f}%" for t in PCK_THRESHOLDS)
            )

    S3_SWEEP_RESULTS[model_key] = rows
    sweep_path = os.path.join(RESULTS_ROOT, f"{model_key}_stage3_val_sweep.csv")
    pd.DataFrame(rows).to_csv(sweep_path, index=False)
    print(f"  Saved val sweep: {sweep_path}")

    # Free sim maps before loading the next backbone
    del sim_records

print("\nVal sweep complete for all backbones.")



  Soft-argmax val sweep: dinov2_vitl14
  Reloaded pretrained weights: dinov2_vitl14
  Loaded fine-tuned weights (N=4): /content/drive/MyDrive/checkpoints/finetuned/dinov2_vitl14_last4.pth
  Phase 1: computing similarity maps (1 pass, no repeats)…


[dinov2_vitl14] sim-maps:   0%|          | 0/5384 [00:00<?, ?it/s]

  Cached 39244 keypoint sim maps in RAM.
  Phase 2: sweeping 16 (temp, window) configs over cached maps…
  temp=0.001   win=3    PCK@0.05=57.11%  PCK@0.1=71.57%  PCK@0.15=77.61%  PCK@0.2=81.41%
  temp=0.001   win=5    PCK@0.05=57.13%  PCK@0.1=71.60%  PCK@0.15=77.63%  PCK@0.2=81.42%
  temp=0.001   win=7    PCK@0.05=57.13%  PCK@0.1=71.62%  PCK@0.15=77.64%  PCK@0.2=81.42%
  temp=0.001   win=11   PCK@0.05=57.12%  PCK@0.1=71.63%  PCK@0.15=77.68%  PCK@0.2=81.42%
  temp=0.005   win=3    PCK@0.05=57.27%  PCK@0.1=71.61%  PCK@0.15=77.62%  PCK@0.2=81.43%
  temp=0.005   win=5    PCK@0.05=57.40%  PCK@0.1=71.71%  PCK@0.15=77.68%  PCK@0.2=81.44%
  temp=0.005   win=7    PCK@0.05=57.41%  PCK@0.1=71.79%  PCK@0.15=77.72%  PCK@0.2=81.46%
  temp=0.005   win=11   PCK@0.05=57.39%  PCK@0.1=71.84%  PCK@0.15=77.81%  PCK@0.2=81.54%
  temp=0.01    win=3    PCK@0.05=57.43%  PCK@0.1=71.66%  PCK@0.15=77.66%  PCK@0.2=81.44%
  temp=0.01    win=5    PCK@0.05=57.63%  PCK@0.1=71.84%  PCK@0.15=77.76%  PCK@0.2=81.46%
  tem

[dinov3_vitb16] sim-maps:   0%|          | 0/5384 [00:00<?, ?it/s]

  Cached 39244 keypoint sim maps in RAM.
  Phase 2: sweeping 16 (temp, window) configs over cached maps…
  temp=0.001   win=3    PCK@0.05=57.26%  PCK@0.1=74.72%  PCK@0.15=81.99%  PCK@0.2=86.03%
  temp=0.001   win=5    PCK@0.05=57.27%  PCK@0.1=74.75%  PCK@0.15=82.01%  PCK@0.2=86.04%
  temp=0.001   win=7    PCK@0.05=57.27%  PCK@0.1=74.77%  PCK@0.15=82.03%  PCK@0.2=86.04%
  temp=0.001   win=11   PCK@0.05=57.26%  PCK@0.1=74.78%  PCK@0.15=82.06%  PCK@0.2=86.05%
  temp=0.005   win=3    PCK@0.05=58.01%  PCK@0.1=74.88%  PCK@0.15=82.07%  PCK@0.2=86.08%
  temp=0.005   win=5    PCK@0.05=58.04%  PCK@0.1=75.01%  PCK@0.15=82.11%  PCK@0.2=86.13%
  temp=0.005   win=7    PCK@0.05=58.06%  PCK@0.1=75.12%  PCK@0.15=82.18%  PCK@0.2=86.16%
  temp=0.005   win=11   PCK@0.05=57.96%  PCK@0.1=75.15%  PCK@0.15=82.30%  PCK@0.2=86.26%
  temp=0.01    win=3    PCK@0.05=58.63%  PCK@0.1=75.08%  PCK@0.15=82.15%  PCK@0.2=86.15%
  temp=0.01    win=5    PCK@0.05=58.72%  PCK@0.1=75.35%  PCK@0.15=82.27%  PCK@0.2=86.22%
  tem

[sam_vit_b_res1024] sim-maps:   0%|          | 0/5384 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [20]:
# ============================================================
# STEP 3 — CELL D: Pick best (temp, window) per backbone on
# the val sweep, then run a single full-test evaluation.
# Compare soft-argmax vs argmax on the same checkpoint.
# ============================================================

S3_BEST_CFG   = {}   # model_key -> best (temp, win)
S3_TEST_ROWS  = []   # rows for the summary CSV

for model_key in MODEL_SPECS:
    rows = S3_SWEEP_RESULTS.get(model_key)
    if not rows:
        # Try loading from disk (if sweep was run in a previous session)
        sweep_path = os.path.join(RESULTS_ROOT, f"{model_key}_stage3_val_sweep.csv")
        if os.path.exists(sweep_path):
            rows = pd.read_csv(sweep_path).to_dict("records")
        else:
            print(f"  No val sweep data for {model_key} — skipping.")
            continue

    # Best config = highest val PCK@0.1
    best = max(rows, key=lambda r: r["PCK@0.1"])
    best_temp = best["temperature"]
    best_win  = int(best["window_size"])
    S3_BEST_CFG[model_key] = {"temperature": best_temp, "window_size": best_win}
    print(f"\n{model_key}: best val config -> temp={best_temp}  win={best_win}  "
          f"(val PCK@0.1={best['PCK@0.1']:.2f}%)")

    load_best_checkpoint(model_key)

    # ── Argmax on best checkpoint (test) ──────────────────────────────────────
    print(f"  Running argmax test eval ...")
    res_argmax = evaluate_cached(
        model_key,
        use_softargmax=False,
        save_csv=True,
        bypass_cache=True,
        max_pairs=None,
        split="test",
    )
    pck_argmax = res_argmax["per_keypoint"]

    # ── Soft-argmax on best checkpoint + best hyper-params (test) ─────────────
    print(f"  Running soft-argmax test eval (temp={best_temp}, win={best_win}) ...")
    res_sa = evaluate_cached(
        model_key,
        use_softargmax=True,
        softargmax_temp=best_temp,
        softargmax_window=best_win,
        save_csv=True,
        bypass_cache=True,
        max_pairs=None,
        split="test",
    )
    pck_sa = res_sa["per_keypoint"]

    for mode, pck in [("argmax", pck_argmax), ("softargmax", pck_sa)]:
        S3_TEST_ROWS.append({
            "model_key": model_key,
            "N_finetuned": BEST_CKPTS[model_key]["N"],
            "mode": mode,
            "temperature": best_temp if mode == "softargmax" else None,
            "window_size": best_win if mode == "softargmax" else None,
            **pck,
        })

# Save summary
s3_summary_path = os.path.join(RESULTS_ROOT, "stage3_summary.csv")
s3_df = pd.DataFrame(S3_TEST_ROWS)
s3_df.to_csv(s3_summary_path, index=False)
print(f"\nSaved Stage 3 summary: {s3_summary_path}")
print(s3_df.to_string(index=False))



dinov2_vitl14: best val config -> temp=0.05  win=7  (val PCK@0.1=72.33%)
  Reloaded pretrained weights: dinov2_vitl14
  Loaded fine-tuned weights (N=4): /content/drive/MyDrive/checkpoints/finetuned/dinov2_vitl14_last4.pth
  Running argmax test eval ...


Evaluating dinov2_vitl14:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 58.12%
  PCK@0.1: 72.85%
  PCK@0.15: 78.87%
  PCK@0.2: 82.29%
Saved per-image CSV: /content/drive/MyDrive/results/dinov2_vitl14-argmax_per_image_results.csv
Saved per-keypoint CSV: /content/drive/MyDrive/results/dinov2_vitl14-argmax_per_keypoint_results.csv
  Running soft-argmax test eval (temp=0.05, win=7) ...


Evaluating dinov2_vitl14:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 59.31%
  PCK@0.1: 73.49%
  PCK@0.15: 79.18%
  PCK@0.2: 82.54%
Saved per-image CSV: /content/drive/MyDrive/results/dinov2_vitl14-softargmax_per_image_results.csv
Saved per-keypoint CSV: /content/drive/MyDrive/results/dinov2_vitl14-softargmax_per_keypoint_results.csv

dinov3_vitb16: best val config -> temp=0.05  win=7  (val PCK@0.1=76.88%)
  Reloaded pretrained weights: dinov3_vitb16
  Loaded fine-tuned weights (N=2): /content/drive/MyDrive/checkpoints/finetuned/dinov3_vitb16_last2.pth
  Running argmax test eval ...


...


Per-keypoint PCK:
  PCK@0.05: 59.27%
  PCK@0.1: 76.41%
  PCK@0.15: 82.76%
  PCK@0.2: 86.01%
Saved per-image CSV: /content/drive/MyDrive/results/dinov3_vitb16-softargmax_per_image_results.csv
Saved per-keypoint CSV: /content/drive/MyDrive/results/dinov3_vitb16-softargmax_per_keypoint_results.csv

sam_vit_b_res1024: best val config -> temp=0.05  win=11  (val PCK@0.1=45.58%)
  Reloaded pretrained weights: sam_vit_b_res1024
  Loaded fine-tuned weights (N=2): /content/drive/MyDrive/checkpoints/finetuned/sam_vit_b_res1024_last2.pth
  Running argmax test eval ...


Evaluating sam_vit_b_res1024:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 33.03%
  PCK@0.1: 44.08%
  PCK@0.15: 51.89%
  PCK@0.2: 58.07%
Saved per-image CSV: /content/drive/MyDrive/results/sam_vit_b_res1024-argmax_per_image_results.csv
Saved per-keypoint CSV: /content/drive/MyDrive/results/sam_vit_b_res1024-argmax_per_keypoint_results.csv
  Running soft-argmax test eval (temp=0.05, win=11) ...


Evaluating sam_vit_b_res1024:   0%|          | 0/12234 [00:00<?, ?it/s]


Per-keypoint PCK:
  PCK@0.05: 33.70%
  PCK@0.1: 46.02%
  PCK@0.15: 53.13%
  PCK@0.2: 58.93%
Saved per-image CSV: /content/drive/MyDrive/results/sam_vit_b_res1024-softargmax_per_image_results.csv
Saved per-keypoint CSV: /content/drive/MyDrive/results/sam_vit_b_res1024-softargmax_per_keypoint_results.csv

Saved Stage 3 summary: /content/drive/MyDrive/results/stage3_summary.csv
        model_key  N_finetuned       mode  temperature  window_size  PCK@0.05   PCK@0.1  PCK@0.15   PCK@0.2
    dinov2_vitl14            4     argmax          NaN          NaN 58.117471 72.850059 78.873064 82.294403
    dinov2_vitl14            4 softargmax         0.05          7.0 59.314147 73.494249 79.182139 82.536681
    dinov3_vitb16            2     argmax          NaN          NaN 56.807581 74.476950 81.332080 85.342134
    dinov3_vitb16            2 softargmax         0.05          7.0 59.272258 76.406123 82.756317 86.014627
sam_vit_b_res1024            2     argmax          NaN          NaN 33.028032 44.

In [21]:
# ============================================================
# STEP 3 — CELL E: Pretty comparison table.
# For each backbone show argmax vs soft-argmax per threshold
# and print the absolute gain (Δ).
# ============================================================

s3_df = pd.read_csv(os.path.join(RESULTS_ROOT, "stage3_summary.csv"))

print("\n" + "="*90)
print("STAGE 3 RESULTS — Argmax vs Window Soft-Argmax (full test split)")
print("="*90)

header = f"{'Model':<28} {'Mode':<14} " + "  ".join(f"PCK@{t}" for t in PCK_THRESHOLDS)
print(header)
print("-" * 90)

for mk in s3_df["model_key"].unique():
    sub = s3_df[s3_df["model_key"] == mk]
    for mode in ["argmax", "softargmax"]:
        row = sub[sub["mode"] == mode]
        if row.empty:
            continue
        row = row.iloc[0]
        label = f"{mk} [{mode}]"
        pck_str = "  ".join(f"{row[f'PCK@{t}']:>8.2f}%" for t in PCK_THRESHOLDS)
        print(f"{label:<42} {pck_str}")

    # Print delta row
    arg_row = sub[sub["mode"] == "argmax"]
    sa_row  = sub[sub["mode"] == "softargmax"]
    if not arg_row.empty and not sa_row.empty:
        arg_row = arg_row.iloc[0]
        sa_row  = sa_row.iloc[0]
        delta_str = "  ".join(
            f"{sa_row[f'PCK@{t}'] - arg_row[f'PCK@{t}']:>+8.2f}%"
            for t in PCK_THRESHOLDS
        )
        cfg = S3_BEST_CFG.get(mk, {})
        print(f"{'  Δ (soft-argmax − argmax)':<42} {delta_str}"
              f"   [temp={cfg.get('temperature','?')}, win={cfg.get('window_size','?')}]")
    print()

print("="*90)
print("\nInterpretation:")
print("  • Positive Δ at PCK@0.05 / 0.1 indicates soft-argmax improves fine-grained accuracy.")
print("  • Benefits are typically larger at tight thresholds (0.05) where sub-pixel")
print("    precision matters most.")

# Sync new results to Drive
sync_features_to_drive()
print("\nAll Stage 3 results synced to Drive.")



STAGE 3 RESULTS — Argmax vs Window Soft-Argmax (full test split)
Model                        Mode           PCK@0.05  PCK@0.1  PCK@0.15  PCK@0.2
------------------------------------------------------------------------------------------
dinov2_vitl14 [argmax]                        58.12%     72.85%     78.87%     82.29%
dinov2_vitl14 [softargmax]                    59.31%     73.49%     79.18%     82.54%
  Δ (soft-argmax − argmax)                    +1.20%     +0.64%     +0.31%     +0.24%   [temp=0.05, win=7]

dinov3_vitb16 [argmax]                        56.81%     74.48%     81.33%     85.34%
dinov3_vitb16 [softargmax]                    59.27%     76.41%     82.76%     86.01%
  Δ (soft-argmax − argmax)                    +2.46%     +1.93%     +1.42%     +0.67%   [temp=0.05, win=7]

sam_vit_b_res1024 [argmax]                    33.03%     44.08%     51.89%     58.07%
sam_vit_b_res1024 [softargmax]                33.70%     46.02%     53.13%     58.93%
  Δ (soft-argmax − argmax)    